# 01 — Node Classification, 1-shot (Table 1)

Reproduces the Node2Graph / 1-shot columns of **Table 1**, using the paper's actual training protocol: for `--dataset X`, `main.py` pretrains on all *other* datasets in `["wisconsin","texas","cornell","cora","citeseer","pubmed","computers","photo","chameleon","squirrel"]` and evaluates transfer on `X` (confirmed in `trainers/node2graph_trainer.py`; this leave-one-dataset-out logic is automatic, not a CLI flag).

Chameleon/Squirrel are excluded as *targets* here (as in the paper's main Table 1, due to known train/test leakage issues in those two datasets — Platonov et al. 2023) but remain available as pretraining sources for the others automatically.

**Prereq.** Run `00_setup.ipynb` in this same Colab session first.

Each dataset gets its own cell so a crash on one (e.g. OOM) doesn't lose progress on the others — just re-run that one cell.

In [ ]:
import os, sys, glob

# Self-sufficient by design: Colab typically gives each notebook its own fresh
# runtime, so REPO_DIR/DATA_DIR/etc from 00_setup.ipynb do NOT carry over unless
# you kept the exact same runtime connected. Re-declare the same paths here and
# fail loudly (with an actionable message) if the actual repo/build aren't
# present in *this* runtime, rather than silently trying to use a missing var.
REPO_DIR = '/content/R-GFM'
BASE = '/content/drive/MyDrive/R-GFM'
DATA_DIR = f'{BASE}/datasets'
CKPT_DIR = f'{BASE}/checkpoints'
RESULTS_DIR = f'{BASE}/results'

if not os.path.isdir(REPO_DIR) or not glob.glob(f'{REPO_DIR}/graph_aug/graph_aug_cuda*.so'):
    raise RuntimeError(
        "R-GFM repo / built CUDA extension not found in this runtime.\n"
        "Run 00_setup.ipynb FIRST in this exact runtime (Runtime -> Manage sessions "
        "to check if it's still connected). If this is a fresh runtime, 00_setup's "
        "clone + pip installs + CUDA build must be redone here — Drive contents "
        "persist across runtimes, but the cloned repo and build artifacts do not."
    )

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')  # no-op if already mounted
for d in [DATA_DIR, CKPT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

NC_LOG_DIR = f'{RESULTS_DIR}/nc_1shot'
os.makedirs(NC_LOG_DIR, exist_ok=True)
print('Logs ->', NC_LOG_DIR)

In [ ]:
# wisconsin
LOG = f'{NC_LOG_DIR}/wisconsin.log'
!python main.py --dataset wisconsin --epochs 150 --shots 1 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_1shot_wisconsin \
    2>&1 | tee $LOG

In [ ]:
# texas
LOG = f'{NC_LOG_DIR}/texas.log'
!python main.py --dataset texas --epochs 150 --shots 1 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_1shot_texas \
    2>&1 | tee $LOG

In [ ]:
# cornell
LOG = f'{NC_LOG_DIR}/cornell.log'
!python main.py --dataset cornell --epochs 150 --shots 1 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_1shot_cornell \
    2>&1 | tee $LOG

In [ ]:
# citeseer
LOG = f'{NC_LOG_DIR}/citeseer.log'
!python main.py --dataset citeseer --epochs 150 --shots 1 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_1shot_citeseer \
    2>&1 | tee $LOG

In [ ]:
# cora
LOG = f'{NC_LOG_DIR}/cora.log'
!python main.py --dataset cora --epochs 150 --shots 1 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_1shot_cora \
    2>&1 | tee $LOG

In [ ]:
# pubmed
LOG = f'{NC_LOG_DIR}/pubmed.log'
!python main.py --dataset pubmed --epochs 150 --shots 1 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_1shot_pubmed \
    2>&1 | tee $LOG

In [ ]:
# computers
LOG = f'{NC_LOG_DIR}/computers.log'
!python main.py --dataset computers --epochs 150 --shots 1 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_1shot_computers \
    2>&1 | tee $LOG

In [ ]:
# photo
LOG = f'{NC_LOG_DIR}/photo.log'
!python main.py --dataset photo --epochs 150 --shots 1 --device 0 \
    --dataset_dir "$DATA_DIR" --checkpoint_dir "$CKPT_DIR" --checkpoint_prefix node2graph_1shot_photo \
    2>&1 | tee $LOG

In [ ]:
# Quick summary — pull the final "Test Accuracy: mean +/- std" line from each log.
import re
for ds in ['wisconsin', 'texas', 'cornell', 'citeseer', 'cora', 'pubmed', 'computers', 'photo']:
    log = f'{NC_LOG_DIR}/{ds}.log'
    if not os.path.exists(log):
        print(f'{ds:12s}  (not run yet)')
        continue
    txt = open(log).read()
    m = re.findall(r'Test Accuracy:\s*([0-9.]+)\s*\+/-\s*([0-9.]+)', txt)
    if m:
        acc, std = m[-1]
        print(f'{ds:12s}  {float(acc)*100:.2f} +/- {float(std)*100:.2f}')
    else:
        print(f'{ds:12s}  (no result line found — check log)')

**Target numbers (paper's Table 1, R-GFM row, 1-shot node classification accuracy %).**

| Wisconsin | Cornell | Citeseer | Cora | Pubmed | Computers | Photos | Texas |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 35.41 ± 7.29 | 36.71 ± 9.92 | 57.54 ± 9.49 | 49.50 ± 3.97 | 49.80 ± 5.38 | 52.30 ± 3.33 | 61.08 ± 5.26 | 32.36 ± 12.10 |

Move on to `02_fewshot.ipynb` for 3-shot/5-shot (Tables 2 & 3), or `03_link_prediction.ipynb`.